# 04 - VAE screening (winter)

Public research notebook for the STARS workflow. Dataset files and generated artifacts are intentionally excluded from Git. See `../data/README.md` before running.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
(DATA_DIR / "raw").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "interim").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "processed").mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT / "src"))


In [ ]:
# full_conv1d_vae_training_fixed.py
import os, gzip, pickle, math, copy, time, csv, numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt

# ---------------------------
# Repro
# ---------------------------
def set_seed(seed=42):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(28)

# ---------------------------
# Load & normalize data
# ---------------------------
with gzip.open(str(DATA_DIR / "interim" / "windows_first2weeksjan.pkl.gz"), "rb") as f:
    windows = pickle.load(f)

window_norm = {}
for bid, w in windows.items():
    w = np.asarray(w, dtype=np.float32)
    if w.ndim == 2:               # multiple windows; z-score each row and pick first
        mu = w.mean(axis=1, keepdims=True)
        sd = w.std(axis=1, keepdims=True) + 1e-6
        w_z = (w - mu) / sd
        w_z = w_z[0]
    else:
        mu = w.mean()
        sd = w.std() + 1e-6
        w_z = (w - mu) / sd
    assert w_z.size == 336, f"{bid}: expected 336 points, got {w_z.size}"
    window_norm[bid] = w_z.astype(np.float32)

win_list, bld_index = [], []
for bid, w in window_norm.items():
    arr = np.asarray(w, dtype=np.float32).ravel()
    assert arr.size == 336
    win_list.append(arr); bld_index.append(bid)

X = np.stack(win_list)                 # (N, 336)
bld_index = np.array(bld_index)
print("X shape:", X.shape)

# ---------------------------
# Dataset
# ---------------------------
class WindowDataset(Dataset):
    def __init__(self, arr): self.arr = torch.from_numpy(arr)
    def __len__(self):      return len(self.arr)
    def __getitem__(self, i): return self.arr[i]

ds = WindowDataset(X)

# ---------------------------
# Conv1D VAE
# ---------------------------
class Conv1DVAE(nn.Module):
    def __init__(self, in_len=336, latent_dim=32, c1=32, c2=64, k1=7, k2=5):
        super().__init__()
        self.in_len = in_len
        self.latent = latent_dim
        self.c1, self.c2 = c1, c2
        self.k1, self.k2 = k1, k2
        p1, p2 = k1//2, k2//2

        self.enc_conv = nn.Sequential(
            nn.Conv1d(1,  c1, kernel_size=k1, stride=1, padding=p1), nn.ReLU(),
            nn.Conv1d(c1, c2, kernel_size=k2, stride=1, padding=p2), nn.ReLU()
        )
        self.flat_dim = c2 * in_len
        self.enc_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flat_dim, 256), nn.ReLU()
        )
        self.mu   = nn.Linear(256, latent_dim)
        self.logv = nn.Linear(256, latent_dim)

        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, self.flat_dim), nn.ReLU()
        )
        self.dec_deconv = nn.Sequential(
            nn.ConvTranspose1d(c2, c1, kernel_size=k2, stride=1, padding=p2), nn.ReLU(),
            nn.ConvTranspose1d(c1, 1,  kernel_size=k1, stride=1, padding=p1)
        )

    def encode(self, x):
        x = x.unsqueeze(1)               # (B, 1, 336)
        h = self.enc_conv(x)             # (B, c2, 336)
        h = self.enc_fc(h)               # (B, 256)
        mu, logv = self.mu(h), self.logv(h)
        return mu, logv

    def reparameterise(self, mu, logv):
        std = torch.exp(0.5 * logv)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.dec_fc(z)                              # (B, c2*336)
        h = h.view(-1, self.c2, self.in_len)           # (B, c2, 336)
        xhat = self.dec_deconv(h)                       # (B, 1, 336)
        return xhat.squeeze(1)                          # (B, 336)

    def forward(self, x):
        mu, logv = self.encode(x)
        z = self.reparameterise(mu, logv)
        recon = self.decode(z)
        return recon, mu, logv

# ---------------------------
# Training utilities
# ---------------------------
def elbo_components(model, xb, mse, beta=1.0):
    recon, mu, logv = model(xb)
    recon_loss = mse(recon, xb)                           # sum
    kl_loss    = -0.5 * torch.sum(1 + logv - mu.pow(2) - logv.exp())
    total      = recon_loss + beta * kl_loss
    return total, recon_loss, kl_loss

class EarlyStopping:
    def __init__(self, patience=20, min_delta=0.0, restore_best=True):
        self.patience, self.min_delta, self.restore_best = patience, min_delta, restore_best
        self.best = math.inf; self.count = 0; self.best_state = None

    def step(self, val_loss, model):
        improved = (self.best - val_loss) > self.min_delta
        if improved:
            self.best = val_loss; self.count = 0
            if self.restore_best: self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.count += 1
        return self.count >= self.patience

    def restore(self, model):
        if self.restore_best and self.best_state is not None:
            model.load_state_dict(self.best_state)

def run_train_val(model, train_loader, val_loader, device,
                  epochs=200, lr=1e-3, beta=1.0, weight_decay=0.0,
                  patience=20, min_delta=0.0, verbose=True,
                  log_csv_path=str(RESULTS_DIR / "training_log.csv")):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    mse = nn.MSELoss(reduction="sum")
    es  = EarlyStopping(patience=patience, min_delta=min_delta, restore_best=True)

    if log_csv_path:
        with open(log_csv_path, "w", newline="") as f:
            csv.writer(f).writerow([
                "epoch","train_total","train_recon","train_kl",
                "val_total","val_recon","val_kl","lr","seconds"
            ])

    best_prev = math.inf
    for ep in range(1, epochs + 1):
        t0 = time.time()
        # ---- train ----
        model.train()
        train_tot = train_rec = train_kl = 0.0; n_train = 0
        for xb in train_loader:
            xb = xb.to(device)
            total, recon_loss, kl_loss = elbo_components(model, xb, mse, beta=beta)
            opt.zero_grad(); total.backward(); opt.step()
            bs = xb.size(0); n_train += bs
            train_tot += total.item(); train_rec += recon_loss.item(); train_kl += kl_loss.item()
        train_tot_ps = train_tot / n_train; train_rec_ps = train_rec / n_train; train_kl_ps = train_kl / n_train

        # ---- valid ----
        model.eval()
        val_tot = val_rec = val_kl = 0.0; n_val = 0
        with torch.no_grad():
            for xb in val_loader:
                xb = xb.to(device)
                total, recon_loss, kl_loss = elbo_components(model, xb, mse, beta=beta)
                bs = xb.size(0); n_val += bs
                val_tot += total.item(); val_rec += recon_loss.item(); val_kl += kl_loss.item()
        val_tot_ps = val_tot / n_val; val_rec_ps = val_rec / n_val; val_kl_ps = val_kl / n_val

        secs = time.time() - t0; lr_now = opt.param_groups[0]["lr"]
        delta = val_tot_ps - best_prev
        if verbose:
            sign = "↘" if delta < 0 else "↗" if delta > 0 else "→"
            print(
                f"epoch {ep:03d} | "
                f"train: total {train_tot_ps:.6f} (recon {train_rec_ps:.6f}, kl {train_kl_ps:.6f}) | "
                f"val: total {val_tot_ps:.6f} (recon {val_rec_ps:.6f}, kl {val_kl_ps:.6f}) | "
                f"best {best_prev:.6f} {sign} Δ{delta:+.6f} | "
                f"patience {es.count}/{es.patience} | lr {lr_now:.1e} | {secs:.1f}s"
            )

        if log_csv_path:
            with open(log_csv_path, "a", newline="") as f:
                csv.writer(f).writerow([
                    ep, train_tot_ps, train_rec_ps, train_kl_ps,
                    val_tot_ps, val_rec_ps, val_kl_ps, lr_now, secs
                ])

        stop = es.step(val_tot_ps, model)
        best_prev = es.best
        if stop:
            if verbose: print(f"Early stopping at epoch {ep}. Restoring best weights.")
            break

    es.restore(model)
    return {"best_val_total": es.best, "log_csv_path": log_csv_path}

# ---------------------------
# Fixed split 80/20
# ---------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
n_total = len(ds); n_train = int(0.8 * n_total); n_val = n_total - n_train
train_set, val_set = random_split(ds, [n_train, n_val],
                                  generator=torch.Generator().manual_seed(123))
print(f"Train: {len(train_set)} | Val: {len(val_set)}")

# ---------------------------
# Fixed hyperparameters (no tuning)
# ---------------------------
fixed_cfg = {'c1': 32, 'k_pair': (5, 3), 'latent_dim': 32, 'lr': 1e-3}
best_k1, best_k2 = fixed_cfg['k_pair']
best_c1 = fixed_cfg['c1']; best_c2 = 2 * best_c1
best_latent = fixed_cfg['latent_dim']; best_lr = fixed_cfg['lr']

# ---------------------------
# Final train on 80%, eval on 20%
# ---------------------------
train_loader = DataLoader(train_set, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False)

model = Conv1DVAE(in_len=336, latent_dim=best_latent, c1=best_c1, c2=best_c2, k1=best_k1, k2=best_k2)
train_info = run_train_val(
    model, train_loader, val_loader, device,
    epochs=200, lr=best_lr, beta=1.0, weight_decay=0.0,
    patience=20, min_delta=0.0, verbose=True, log_csv_path=str(RESULTS_DIR / "training_log_jan.csv")
)
print(f"\nFinal held-out VAL (best total per-sample): {train_info['best_val_total']:.6f}")

# ---------------------------
# Per-sample MSE over whole set, worst-case plot, quantiles
# ---------------------------
model.eval()
full_loader = DataLoader(ds, batch_size=1024, shuffle=False)
all_mse = []
with torch.no_grad():
    for xb in full_loader:
        xb = xb.to(device)
        recon, _, _ = model(xb)
        all_mse.append(torch.mean((recon - xb)**2, dim=1).cpu())
mse_per_sample = torch.cat(all_mse).numpy()

idx_worst = int(np.argmax(mse_per_sample))
worst_bid = bld_index[idx_worst]
max_mse   = float(mse_per_sample[idx_worst])
print(f"\nWorst reconstruction: idx={idx_worst}, building_id={worst_bid}, MSE={max_mse:.6f}")

p95 = np.percentile(mse_per_sample, 95)
p99 = np.percentile(mse_per_sample, 99)
print(f"P95 MSE: {p95:.6f} | P99 MSE: {p99:.6f}")
out_idx = np.where(mse_per_sample >= p99)[0]
print(f"Top 1% outlier count: {len(out_idx)}")
print("Example outlier building_ids:", bld_index[out_idx[:10]])

In [ ]:
import os
import matplotlib.pyplot as plt

if os.path.exists(str(RESULTS_DIR / "training_log_jan.csv")):
    import pandas as pd
    df = pd.read_csv(str(RESULTS_DIR / "training_log_jan.csv"))

    plt.figure(figsize=(9, 5.5))
    plt.plot(df["epoch"], df["train_total"], label="train", linewidth=2.5)
    plt.plot(df["epoch"], df["val_total"],   label="test",   linewidth=2.5)

    # Bigger, bolder axis labels
    plt.xlabel("Epoch", fontsize=20, labelpad=10)
    plt.ylabel("ELBO per-sample", fontsize=20, labelpad=10)

    ax = plt.gca()

    # Bigger tick labels and thicker ticks/axes
    ax.tick_params(axis="both", labelsize=18, width=2, length=7)
    for spine in ax.spines.values():
        spine.set_linewidth(2)

    # Larger legend and title (optional)
    plt.legend(fontsize=18)
    # plt.title("VAE ELBO Learning Curves", fontsize=20, fontweight="bold")

    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# full_fc_vae_training.py
import os, gzip, pickle, math, copy, time, csv, numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt

# ---------------------------
# Repro
# ---------------------------
def set_seed(seed=42):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

# ---------------------------
# Load & normalize data
# ---------------------------
with gzip.open(str(DATA_DIR / "interim" / "windows_first2weeksjan.pkl.gz"), "rb") as f:
    windows = pickle.load(f)

window_norm = {}
for bid, w in windows.items():
    w = np.asarray(w, dtype=np.float32)
    if w.ndim == 2:               # multiple windows; z-score each row and pick first
        mu = w.mean(axis=1, keepdims=True)
        sd = w.std(axis=1, keepdims=True) + 1e-6
        w_z = (w - mu) / sd
        w_z = w_z[0]
    else:
        mu = w.mean()
        sd = w.std() + 1e-6
        w_z = (w - mu) / sd
    assert w_z.size == 336, f"{bid}: expected 336 points, got {w_z.size}"
    window_norm[bid] = w_z.astype(np.float32)

win_list, bld_index = [], []
for bid, w in window_norm.items():
    arr = np.asarray(w, dtype=np.float32).ravel()
    assert arr.size == 336
    win_list.append(arr); bld_index.append(bid)

X = np.stack(win_list)                 # (N, 336)
bld_index = np.array(bld_index)
print("X shape:", X.shape)

# ---------------------------
# Dataset
# ---------------------------
class WindowDataset(Dataset):
    def __init__(self, arr): self.arr = torch.from_numpy(arr)
    def __len__(self):      return len(self.arr)
    def __getitem__(self, i): return self.arr[i]

ds = WindowDataset(X)

# ----------------------------------------------------------------------
# 2)  Simple 1-D fully-connected VAE  (your spec)
# ----------------------------------------------------------------------
LATENT_DIM = 16   # size of representation

class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(336, 128), nn.ReLU(),
            nn.Linear(128, 64),  nn.ReLU()
        )
        self.mu   = nn.Linear(64, LATENT_DIM)
        self.logv = nn.Linear(64, LATENT_DIM)

        self.dec = nn.Sequential(
            nn.Linear(LATENT_DIM, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 336)             # reconstruct full window
        )

    def encode(self, x):
        h   = self.enc(x)
        mu  = self.mu(h)
        logv= self.logv(h)
        return mu, logv

    def reparameterise(self, mu, logv):
        std = torch.exp(0.5 * logv)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logv = self.encode(x)
        z        = self.reparameterise(mu, logv)
        recon    = self.dec(z)
        return recon, mu, logv

# ---------------------------
# Training utilities (unchanged)
# ---------------------------
def elbo_components(model, xb, mse, beta=1.0):
    recon, mu, logv = model(xb)
    recon_loss = mse(recon, xb)                           # sum over dims
    kl_loss    = -0.5 * torch.sum(1 + logv - mu.pow(2) - logv.exp())
    total      = recon_loss + beta * kl_loss
    return total, recon_loss, kl_loss

class EarlyStopping:
    def __init__(self, patience=20, min_delta=0.0, restore_best=True):
        self.patience, self.min_delta, self.restore_best = patience, min_delta, restore_best
        self.best = math.inf; self.count = 0; self.best_state = None

    def step(self, val_loss, model):
        improved = (self.best - val_loss) > self.min_delta
        if improved:
            self.best = val_loss; self.count = 0
            if self.restore_best: self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.count += 1
        return self.count >= self.patience

    def restore(self, model):
        if self.restore_best and self.best_state is not None:
            model.load_state_dict(self.best_state)

def run_train_val(model, train_loader, val_loader, device,
                  epochs=200, lr=1e-3, beta=1.0, weight_decay=0.0,
                  patience=20, min_delta=0.0, verbose=True,
                  log_csv_path=str(RESULTS_DIR / "training_log.csv")):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    mse = nn.MSELoss(reduction="sum")
    es  = EarlyStopping(patience=patience, min_delta=min_delta, restore_best=True)

    if log_csv_path:
        with open(log_csv_path, "w", newline="") as f:
            csv.writer(f).writerow([
                "epoch","train_total","train_recon","train_kl",
                "val_total","val_recon","val_kl","lr","seconds"
            ])

    best_prev = math.inf
    for ep in range(1, epochs + 1):
        t0 = time.time()
        # ---- train ----
        model.train()
        train_tot = train_rec = train_kl = 0.0; n_train = 0
        for xb in train_loader:
            xb = xb.to(device)
            total, recon_loss, kl_loss = elbo_components(model, xb, mse, beta=beta)
            opt.zero_grad(); total.backward(); opt.step()
            bs = xb.size(0); n_train += bs
            train_tot += total.item(); train_rec += recon_loss.item(); train_kl += kl_loss.item()
        train_tot_ps = train_tot / n_train; train_rec_ps = train_rec / n_train; train_kl_ps = train_kl / n_train

        # ---- valid ----
        model.eval()
        val_tot = val_rec = val_kl = 0.0; n_val = 0
        with torch.no_grad():
            for xb in val_loader:
                xb = xb.to(device)
                total, recon_loss, kl_loss = elbo_components(model, xb, mse, beta=beta)
                bs = xb.size(0); n_val += bs
                val_tot += total.item(); val_rec += recon_loss.item(); val_kl += kl_loss.item()
        val_tot_ps = val_tot / n_val; val_rec_ps = val_rec / n_val; val_kl_ps = val_kl / n_val

        secs = time.time() - t0; lr_now = opt.param_groups[0]["lr"]
        delta = val_tot_ps - best_prev
        if verbose:
            sign = "↘" if delta < 0 else "↗" if delta > 0 else "→"
            print(
                f"epoch {ep:03d} | "
                f"train: total {train_tot_ps:.6f} (recon {train_rec_ps:.6f}, kl {train_kl_ps:.6f}) | "
                f"val: total {val_tot_ps:.6f} (recon {val_rec_ps:.6f}, kl {val_kl_ps:.6f}) | "
                f"best {best_prev:.6f} {sign} Δ{delta:+.6f} | "
                f"patience {es.count}/{es.patience} | lr {lr_now:.1e} | {secs:.1f}s"
            )

        if log_csv_path:
            with open(log_csv_path, "a", newline="") as f:
                csv.writer(f).writerow([
                    ep, train_tot_ps, train_rec_ps, train_kl_ps,
                    val_tot_ps, val_rec_ps, val_kl_ps, lr_now, secs
                ])

        stop = es.step(val_tot_ps, model)
        best_prev = es.best
        if stop:
            if verbose: print(f"Early stopping at epoch {ep}. Restoring best weights.")
            break

    es.restore(model)
    return {"best_val_total": es.best, "log_csv_path": log_csv_path}

# ---------------------------
# Split 80/20
# ---------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
n_total = len(ds); n_train = int(0.8 * n_total); n_val = n_total - n_train
train_set, val_set = random_split(ds, [n_train, n_val],
                                  generator=torch.Generator().manual_seed(123))
print(f"Train: {len(train_set)} | Val: {len(val_set)}")

# ---------------------------
# Final train (fixed config, no CV)
# ---------------------------
train_loader = DataLoader(train_set, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False)

model = VAE()
train_info = run_train_val(
    model, train_loader, val_loader, device,
    epochs=200, lr=1e-3, beta=1.0, weight_decay=0.0,
    patience=20, min_delta=0.0, verbose=True, log_csv_path=str(RESULTS_DIR / "training_log.csv")
)
print(f"\nFinal held-out VAL (best total per-sample): {train_info['best_val_total']:.6f}")

# ---------------------------
# Per-sample MSE over whole set, worst-case plot, quantiles
# ---------------------------
model.eval()
full_loader = DataLoader(ds, batch_size=1024, shuffle=False)
all_mse = []
with torch.no_grad():
    for xb in full_loader:
        xb = xb.to(device)
        recon, _, _ = model(xb)
        all_mse.append(torch.mean((recon - xb)**2, dim=1).cpu())
mse_per_sample = torch.cat(all_mse).numpy()

idx_worst = int(np.argmax(mse_per_sample))
worst_bid = bld_index[idx_worst]
max_mse   = float(mse_per_sample[idx_worst])
print(f"\nWorst reconstruction: idx={idx_worst}, building_id={worst_bid}, MSE={max_mse:.6f}")

with torch.no_grad():
    x_orig = torch.from_numpy(X[idx_worst]).float().to(device)
    recon  = model(x_orig.unsqueeze(0))[0].squeeze(0).cpu().numpy()

p95 = np.percentile(mse_per_sample, 95)
p99 = np.percentile(mse_per_sample, 99)
print(f"P95 MSE: {p95:.6f} | P99 MSE: {p99:.6f}")
out_idx = np.where(mse_per_sample >= p99)[0]
print(f"Top 1% outlier count: {len(out_idx)}")
print("Example outlier building_ids:", bld_index[out_idx[:10]])

In [ ]:
# full_fc_vae_training_fullset_200ep.py
import os, gzip, pickle, time, csv, math, numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# ---------------------------
# Repro
# ---------------------------
def set_seed(seed=42):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

# ---------------------------
# Load & normalize data
# ---------------------------
with gzip.open(str(DATA_DIR / "interim" / "windows_first2weeksjan.pkl.gz"), "rb") as f:
    windows = pickle.load(f)

window_norm = {}
for bid, w in windows.items():
    w = np.asarray(w, dtype=np.float32)
    if w.ndim == 2:               # multiple windows; z-score each row and pick first
        mu = w.mean(axis=1, keepdims=True)
        sd = w.std(axis=1, keepdims=True) + 1e-6
        w_z = (w - mu) / sd
        w_z = w_z[0]
    else:
        mu = w.mean()
        sd = w.std() + 1e-6
        w_z = (w - mu) / sd
    assert w_z.size == 336, f"{bid}: expected 336 points, got {w_z.size}"
    window_norm[bid] = w_z.astype(np.float32)

win_list, bld_index = [], []
for bid, w in window_norm.items():
    arr = np.asarray(w, dtype=np.float32).ravel()
    assert arr.size == 336
    win_list.append(arr); bld_index.append(bid)

X = np.stack(win_list)                 # (N, 336)
bld_index = np.array(bld_index)
print("X shape:", X.shape)

# ---------------------------
# Dataset
# ---------------------------
class WindowDataset(Dataset):
    def __init__(self, arr): self.arr = torch.from_numpy(arr)
    def __len__(self):      return len(self.arr)
    def __getitem__(self, i): return self.arr[i]

ds = WindowDataset(X)

# ----------------------------------------------------------------------
# Fully-connected VAE
# ----------------------------------------------------------------------
LATENT_DIM = 16

class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(336, 128), nn.ReLU(),
            nn.Linear(128, 64),  nn.ReLU()
        )
        self.mu   = nn.Linear(64, LATENT_DIM)
        self.logv = nn.Linear(64, LATENT_DIM)
        self.dec  = nn.Sequential(
            nn.Linear(LATENT_DIM, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 336)
        )

    def encode(self, x):
        h   = self.enc(x)
        mu  = self.mu(h)
        logv= self.logv(h)
        return mu, logv

    def reparameterise(self, mu, logv):
        std = torch.exp(0.5 * logv)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logv = self.encode(x)
        z        = self.reparameterise(mu, logv)
        recon    = self.dec(z)
        return recon, mu, logv

# ---------------------------
# Training (full dataset, fixed 200 epochs)
# ---------------------------
def elbo_components(model, xb, mse, beta=1.0):
    recon, mu, logv = model(xb)
    recon_loss = mse(recon, xb)                           # sum over dims
    kl_loss    = -0.5 * torch.sum(1 + logv - mu.pow(2) - logv.exp())
    total      = recon_loss + beta * kl_loss
    return total, recon_loss, kl_loss

def train_full(model, train_loader, device,
               epochs=200, lr=1e-3, beta=1.0, weight_decay=0.0,
               verbose=True, log_csv_path=str(RESULTS_DIR / "training_log.csv")):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    mse = nn.MSELoss(reduction="sum")

    if log_csv_path:
        with open(log_csv_path, "w", newline="") as f:
            csv.writer(f).writerow(["epoch","train_total","train_recon","train_kl","lr","seconds"])

    for ep in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        train_tot = train_rec = train_kl = 0.0; n_train = 0
        for xb in train_loader:
            xb = xb.to(device)
            total, recon_loss, kl_loss = elbo_components(model, xb, mse, beta=beta)
            opt.zero_grad(); total.backward(); opt.step()
            bs = xb.size(0); n_train += bs
            train_tot += total.item(); train_rec += recon_loss.item(); train_kl += kl_loss.item()

        train_tot_ps = train_tot / n_train
        train_rec_ps = train_rec / n_train
        train_kl_ps  = train_kl  / n_train
        secs = time.time() - t0; lr_now = opt.param_groups[0]["lr"]

        if verbose:
            print(
                f"epoch {ep:03d} | train total {train_tot_ps:.6f} "
                f"(recon {train_rec_ps:.6f}, kl {train_kl_ps:.6f}) | "
                f"lr {lr_now:.1e} | {secs:.1f}s"
            )

        if log_csv_path:
            with open(log_csv_path, "a", newline="") as f:
                csv.writer(f).writerow([ep, train_tot_ps, train_rec_ps, train_kl_ps, lr_now, secs])

    return {"epochs": epochs, "log_csv_path": log_csv_path}

# ---------------------------
# Run training on FULL dataset (no split)
# ---------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 256
train_loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

model = VAE()
train_info = train_full(
    model, train_loader, device,
    epochs=200, lr=1e-3, beta=1.0, weight_decay=0.0,
    verbose=True, log_csv_path=str(RESULTS_DIR / "training_log.csv")
)
print(f"\nTraining completed for {train_info['epochs']} epochs on full dataset.")

# ---------------------------
# Per-sample MSE over whole set, worst-case, quantiles
# ---------------------------
model.eval()
full_loader = DataLoader(ds, batch_size=1024, shuffle=False)
all_mse = []
with torch.no_grad():
    for xb in full_loader:
        xb = xb.to(device)
        recon, _, _ = model(xb)
        all_mse.append(torch.mean((recon - xb)**2, dim=1).cpu())
mse_per_sample = torch.cat(all_mse).numpy()

idx_worst = int(np.argmax(mse_per_sample))
worst_bid = bld_index[idx_worst]
max_mse   = float(mse_per_sample[idx_worst])
print(f"\nWorst reconstruction: idx={idx_worst}, building_id={worst_bid}, MSE={max_mse:.6f}")

# Plot worst reconstruction (optional)
with torch.no_grad():
    x_orig = torch.from_numpy(X[idx_worst]).float().to(device)
    recon  = model(x_orig.unsqueeze(0))[0].squeeze(0).cpu().numpy()

p95 = np.percentile(mse_per_sample, 95)
p99 = np.percentile(mse_per_sample, 99)
print(f"P95 MSE: {p95:.6f} | P99 MSE: {p99:.6f}")
out_idx = np.where(mse_per_sample >= p99)[0]
print(f"Top 1% outlier count: {len(out_idx)}")
print("Example outlier building_ids:", bld_index[out_idx[:10]])

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4-week (28-day) windows starting every Monday 00:00
# ─────────────────────────────────────────────────────────────
import pandas as pd, numpy as np

CSV_PATH  = str(DATA_DIR / "processed" / "bdg2_ny_2017_with_weather.csv")
TARGETCOL = "electricity_kWh"
HIST_LEN  = 2 * 7 * 24          # 2 weeks × 7 days × 24 h = 336

df = pd.read_csv(CSV_PATH, parse_dates=["timestamp"])

In [ ]:
def build_first2weeks_july_monday(df,
                                  hist_len=336,
                                  col="electricity_kWh"):
    """
    Return, for each building, the first 14-day window that
    • starts at Monday 00:00 inside July
    • contains no NaNs in `col`
    • is contiguous hourly data.

    ----------
    Returns
    ----------
    win_dict : dict[building_id] -> np.ndarray (hist_len,) float32
    counts   : dict[building_id] -> 1    (if window present)
    """
    win_dict, counts = {}, {}

    # pre-filter July rows for speed
    july_mask = (df["month_of_year"] == 1)
    df_july   = df.loc[july_mask].copy()

    for bid, grp in df_july.groupby("building_id"):
        grp = grp.sort_values("timestamp").reset_index(drop=True)

        # find the first Monday 00:00 in this building’s July data
        anchors = grp.index[
            (grp["day_of_week"] == 0) & (grp["hour_of_day"] == 0)
        ]
        if anchors.empty:
            continue                         # no Monday midnight in July

        start = anchors[0]
        end   = start + hist_len
        if end > len(grp):
            continue                         # not enough hours

        window = grp.iloc[start:end]
        xs = window[col].to_numpy(dtype=np.float32)

        # basic quality checks
        if np.isnan(xs).any():
            continue
        elapsed = (window["timestamp"].iloc[-1] -
                   window["timestamp"].iloc[0]).total_seconds()
        if elapsed != (hist_len - 1) * 3600:
            continue                         # missing timestamps

        win_dict[bid] = xs
        counts[bid]   = 1

    return win_dict, counts

In [ ]:
windows_test, counts = build_first2weeks_july_monday(df, hist_len=336)
print(f"Collected {len(windows)} buildings with a clean 2-week July window.")

In [ ]:
import numpy as np
# windows : {building_id: [window₀, window₁, …]}   (each window is 1-D length-336)

window_norm_test   = {}          # same keys, z-scored windows

for bid, w in windows_test.items():
    norm_list   = []
    stats_list  = []

    w   = np.asarray(w, dtype=np.float32)
    μ   = w.mean()
    σ   = w.std() + 1e-6                    # avoid /0
    w_z = (w - μ) / σ                       # z-score

    window_norm_test[bid]   = w_z

# ----------------------------------------------------------------------
# 0)  window_norm:  {building_id: [window₀, window₁, …]}  (each window len-336)
# ----------------------------------------------------------------------
win_list_test  = []
bld_index_test = []

for bid, w in window_norm_test.items():
    arr = np.asarray(w, dtype=np.float32).ravel()  # flatten just in case
    assert arr.size == 336, f"{bid}: expected 336 points, got {arr.size}"
    win_list_test.append(arr)
    bld_index_test.append(bid)

X_test         = np.stack(win_list_test)        # (N_buildings , 336)
bld_index_test = np.array(bld_index_test)

print("X shape :", X_test.shape)           # (N, 336)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

ds  = TensorDataset(torch.from_numpy(X_test))
dl  = DataLoader(ds, batch_size=512, shuffle=False)

latents = []
model.eval()
with torch.no_grad():
    for (xb,) in dl:
        xb   = xb.to(device)
        mu, _ = model.encode(xb)
        latents.append(mu.cpu())

win_repr_test = torch.cat(latents).numpy()      # (N, 16)

In [ ]:
import torch
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Assumes you already have:
#   • bld_index_test  – list/array of building IDs  (len = N)
#   • X_test          – original windows, shape (N, 336)
#   • win_repr_test   – latent codes,   shape (N, latent_dim)
#   • vae             – trained VAE with .dec() decoder
# ------------------------------------------------------------------

results = []
for i, bld_id in enumerate(bld_index_test):
    # original 2-week load
    ts = X_test[i]

    # reconstruction from latent code
    with torch.no_grad():
        recon = (
            model.decode(torch.from_numpy(win_repr_test[i][None, :]).float().to(device))
               .cpu()
               .numpy()
               .ravel()
        )

    # mean-squared reconstruction error
    mse = np.mean((ts - recon) ** 2)
    results.append((bld_id, mse))

# rank from best (lowest error) to worst
df = (
    pd.DataFrame(results, columns=["building_id", "mse"])
      .sort_values("mse", ascending=True)
      .reset_index(drop=True)
)

# save to CSV
df.to_csv(str(RESULTS_DIR / "reconstruction_mse_rank_jan_3.csv"), index=False)

print(df.head())        # quick sanity check

In [ ]:
# ─────────────────────────────────────────────────────────────
# 0)  Imports  (install igraph / leidenalg if you haven't)
# ─────────────────────────────────────────────────────────────
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances

# ─────────────────────────────────────────────────────────────
# 2)  Pick a random sample and find its 5 nearest neighbours
# ─────────────────────────────────────────────────────────────
np.random.seed(0)
idx_sel   = np.random.randint(len(win_repr_test))
latent_sel= X_test[idx_sel][None, :]                     # (1, 16)

# pairwise distance to all points
dists = pairwise_distances(latent_sel, win_list, metric="euclidean").ravel()
nn_ids = dists.argsort()[1:6]                              # 5 closest, skip itself

print("Selected bldg :", bld_index_test[idx_sel])
print("Nearest 5     :", bld_index[nn_ids])

orig = X_test[idx_sel]                                             # (336,)

# 4) Plot – keep both panels on the same y-axis scale
fig, axes = plt.subplots(
    2, 1, figsize=(10, 6),
    sharex=True,         # already sharing x
    sharey=True          # **share y as well**
)

# (a) selected window
axes[0].plot(orig, label="Original", lw=1.8)
axes[0].set_title(f"Building {bld_index_test[idx_sel]} • 4-week window (336 h)")
axes[0].legend()

# (b) 5 nearest neighbours
for j, nb in enumerate(nn_ids, 1):
    axes[1].plot(X[nb], lw=1, label=f"NN{j}: {bld_index[nb]}")
axes[1].set_title("Nearest-neighbour windows in raw space")
axes[1].set_xlabel("Hour")
axes[1].legend(ncol=3, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# 1) full distance matrix  (→ shape = N_test × N_train)
dist_mat = pairwise_distances(X_test, win_list, metric="euclidean")

# 2) sort each row to get indices of nearest-to-farthest training buildings
rank_mat = dist_mat.argsort(axis=1)          # same shape as dist_mat

# 3) build the dictionary
knn_dict = {
    bld_index_test[i]: list(bld_index[rank_mat[i]])
    for i in range(len(bld_index_test))
}

In [ ]:
# distances aligned with each nearest-ID list
knn_dist = {
    bld_index_test[i]: dist_mat[i, rank_mat[i]].tolist()   # convert to plain list
    for i in range(len(bld_index_test))
}

In [ ]:
bld_index_test[1]

In [ ]:
# -------------------------------------------------------------
# 1) Pick the first key in the dict
test_id = 'Cockatoo_assembly_Ed'             # first test building ID
train_ids_sorted = knn_dict[test_id]         # list of training IDs, nearest→far

# indices
idx_test = np.where(bld_index_test == test_id)[0][0]
idx_train_top5 = [np.where(bld_index == tid)[0][0] for tid in train_ids_sorted[:5]]

# series
ts_test = X_test[idx_test]                   # (336,)
ts_neigh = X[idx_train_top5]                 # (5, 336)

# -------------------------------------------------------------
# 4) Plot
# -------------------------------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                         gridspec_kw=dict(height_ratios=[3, 2]))

# (a) original vs reconstruction
axes[0].plot(ts_test, lw=1.8, label=f"Test: {test_id}")
axes[0].set_title(f"Test building {test_id} • 2‑week window (336 h)")
axes[0].legend()

# (b) top‑5 neighbours
for j, (tid, series) in enumerate(zip(train_ids_sorted[:5], ts_neigh), 1):
    axes[1].plot(series, lw=1, label=f"NN{j}: {tid}")
axes[1].set_title("Top‑5 nearest neighbours in training set")
axes[1].set_xlabel("Hour")
axes[1].legend(ncol=3, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import json, os

# ─────────────────────────────────────────────────────────────
# Save knn_dict → JSON  (keys and values cast to str for safety)
# ─────────────────────────────────────────────────────────────
file_path = str(DATA_DIR / "interim" / "knn_dict_raw_1.json")      # change if you prefer a different name
with open(file_path, "w") as f:
    json.dump({str(k): [str(v) for v in vals] for k, vals in knn_dict.items()},
              f, indent=2)

print(f"✔ knn_dict_raw saved to {file_path}")